# Setup

In [1]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
from itertools import permutations, combinations

from seq_utils import create_delay_sequence, fix_overunder_repeats

# Helpers

In [2]:
def shuffle_along_axis(arr, axis):
    idx = np.random.rand(*arr.shape).argsort(axis=axis)
    return np.take_along_axis(arr, idx, axis=axis)

def shuffled(arr):
    arr_shuffled = arr.copy()
    np.random.shuffle(arr_shuffled)
    return arr_shuffled

def geerante_sequence_with_fixed_repeats(ns, stim_iter_per_block):
    wrong_seq = True
    nextseq = []
    while wrong_seq:
        nextseq = create_delay_sequence(stim_iter_per_block-1, ns)+1
        nextseq = fix_overunder_repeats(nextseq, stim_iter_per_block)
        _, counts = np.unique(nextseq, return_counts=True)
        wrong_seq = not np.all(counts == stim_iter_per_block)
    
    return nextseq


def repeat_numbers_for_set_size(set_size, nA, rng):
    """
    Repeat numbers 0..nA-1, each appearing set_size // nA times, as a 1D np.array.
    """
    reps = set_size // nA
    numbers = np.tile(np.arange(nA), reps)
    return rng.permutation(numbers)

def has_consecutive_identical(arr, max_run):
    """
    Returns True if any number repeats 'max_run' or more times in a row.
    """
    if len(arr) < max_run:
        return False
    
    # Check where elements differ from the previous element
    # Prepend True so the first element starts a run
    diffs = np.diff(arr) != 0
    change_indices = np.concatenate(([0], np.where(diffs)[0] + 1, [len(arr)]))
    run_lengths = np.diff(change_indices)
    
    return np.any(run_lengths >= max_run)    

def is_new_sequence(cand, seqs):
    """True if cand is not identical to any sequence in seqs."""
    cand = np.asarray(cand)
    return all(not np.array_equal(cand, s) for s in seqs)

def generate_balanced_random_list(num_seq, set_size, nA, rng):
    """
    Generate a list of length X using numbers 1..nA, 
    each appearing as equally as possible, order randomized.
    """
    all_seqs = []
    for _ in range(num_seq):
        seq = repeat_numbers_for_set_size(set_size, nA, rng)
        while has_consecutive_identical(seq, nA) or not is_new_sequence(seq, all_seqs):
            seq = repeat_numbers_for_set_size(set_size, nA, rng)
        
        all_seqs.append(seq)

    return all_seqs


# Shaped sequence generation

In [3]:
def make_setsize_sequence(
    ns,
    nA,
    conditions,
    rng,
    img_folders,
    n_total_iter=8,
    n_shape_iter=4,
):
    """Match the MATLAB loop: 4 conditions × NB blocks, train then full-set test."""
    rng = np.random.default_rng(rng)
    stimuli = np.arange(1, ns + 1)
    all_st_to_cor_key = generate_balanced_random_list(len(conditions), ns, nA, rng)
    # starts with 2 two because the 1st folder is practice.
    # img_folder_list = rng.choice(np.arange(2, NUM_IMAGE_FOLDERS+1), size=3, replace=False)
    rows = []
    for block, condition in enumerate(conditions):
        stimseq, period_setsize = _train_period(condition, stimuli, ns, n_shape_iter)
        n_test_iter = n_total_iter - n_shape_iter

        nextseq = geerante_sequence_with_fixed_repeats(ns, n_test_iter)
        _, counts = np.unique(nextseq, return_counts=True)
        wrong_seq = not np.all(counts == n_test_iter)
        if wrong_seq:
            print(f"wrong sequence in block {block+1}, condition {condition}")

        stimseq = np.concatenate([stimseq, nextseq])
        # setsize = np.concatenate([setsize, np.full(len(nextseq), ns)])
        iteration = np.zeros_like(stimseq)
        for s in stimuli:
            idx = np.flatnonzero(stimseq == s)
            iteration[idx] = np.arange(1, len(idx) + 1)

        n_train_trials = n_shape_iter * ns
        n_test_trials = ns * n_test_iter
        period = np.concatenate([period_setsize, np.full(n_test_trials, ns)])
        print(n_train_trials, n_test_trials)
        print(len(stimseq), len(iteration), len(period))
        rows.append(
            pd.DataFrame(
                {
                    "block": block + 1,
                    "set_size": ns,
                    "stim": stimseq,
                    "condition": condition,
                    "correct_key": all_st_to_cor_key[block][stimseq - 1],
                    "img_folder": img_folders[block],
                    "period_sz": period,
                    "iteration": iteration,
                }
            )
        )
    return pd.concat(rows, ignore_index=True)


def _train_period(condition, stimuli, ns, n_train):
    if condition == 1:
        # one stimulus at a time, repeated n_train times
        stimseq = np.repeat(stimuli, n_train)
        setsize = np.ones(ns * n_train, dtype=int)
    elif condition == 2:
        chunks = [np.tile(stimuli[k : k + 2], n_train) for k in range(0, ns, 2)]
        stimseq = np.concatenate(chunks)
        setsize = np.full(ns * n_train, 2)
    elif condition == 3:
        # triplets (MATLAB case 2 after the set-size-2 code was commented out)
        chunks = [np.tile(stimuli[k : k + 3], n_train) for k in range(0, ns, 3)]
        stimseq = np.concatenate(chunks)
        setsize = np.full(ns * n_train, 3)
    elif condition == 6:
        chunks = [np.tile(stimuli[k : k + 6], n_train) for k in range(0, ns, 6)]
        stimseq = np.concatenate(chunks)
        setsize = np.full(ns * n_train, 6)
    elif condition == (ns + 1):
        # stimseq = create_delay_sequence(n_train-1, ns)+1
        stimseq = geerante_sequence_with_fixed_repeats(ns, n_train)
        setsize = np.full(ns * n_train, ns)
    else:
        raise ValueError(f"unknown condition {condition}")

    return stimseq, setsize


def get_random_seq_by_perms(sequence, num_seq=10):
    perms = list(permutations(sequence))
    all_perms_seq = list(permutations(perms, len(sequence)))
    # randomly draw num_seq from all_perms_seq
    rng = np.random.default_rng(123)
    idx = rng.choice(len(all_perms_seq), size=num_seq, replace=False)
    chosen_perms = [all_perms_seq[i] for i in idx]
    return [[x for p in triple for x in p] for triple in chosen_perms]

In [6]:
np.arange(2, 10+1)

array([ 2,  3,  4,  5,  6,  7,  8,  9, 10])

In [17]:
DIR_NAME = "/Users/ccnlab/Development/sequences/shaping/v1_rlwm"
exp_type = "v1_rlwm"
NUM_SEQ = 10
NUM_IMAGE_FOLDERS = 10

# TODO randomize the image folder later
IMAGE_FOLDERS = [[2, 3, 4], [5, 6, 7], [3, 4, 5], [5, 6, 2]]

set_sz = 12
rand_cond = set_sz + 1
num_actions = 3
OUTPUT_COL_ORDER = [
    "stim",
    "correct_key",
    "set_size",
    "block",
    "img_folder",
    "condition",
    "period_sz",
    "iteration",
]

all_blocks = get_random_seq_by_perms([2, 3, rand_cond], NUM_SEQ)
rng = np.random.default_rng(1)
folders = np.arange(2, NUM_IMAGE_FOLDERS + 1)

reps = int(np.ceil(NUM_SEQ / len(folders)))
starts = np.concatenate([rng.permutation(folders) for _ in range(reps)])[:NUM_SEQ]
# 2. Prepend each start to a shuffled version of the remaining folders
all_img_folders = np.array(
    [np.r_[s, rng.permutation(folders[folders != s])] for s in starts]
)
all_img_folders

array([[ 9,  3,  5,  2,  8,  6,  7, 10,  4],
       [ 2,  6,  4,  9, 10,  7,  8,  3,  5],
       [ 3,  6, 10,  5,  2,  7,  9,  8,  4],
       [ 6, 10,  5,  4,  7,  2,  9,  8,  3],
       [ 4,  2,  6, 10,  9,  8,  3,  7,  5],
       [ 7,  5,  4, 10,  6,  8,  2,  3,  9],
       [10,  2,  3,  6,  7,  5,  4,  8,  9],
       [ 8,  7,  2,  3,  6, 10,  4,  5,  9],
       [ 5,  8,  2,  9,  6,  3, 10,  7,  4],
       [ 3,  4,  9,  6, 10,  8,  5,  2,  7]])

In [18]:
for i, conds in enumerate(all_blocks):
    rng = np.random.default_rng(i * 123)
    df = make_setsize_sequence(set_sz, num_actions, conds, rng, all_img_folders[i])
    df[OUTPUT_COL_ORDER].to_csv(f"{DIR_NAME}/seq{i+1}_learning.csv", index=False)

48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 96 96
48 48
96 9

# RLWM original seq generation

In [ ]:
num_conditions = 2  # number of different file sequences to generate (change if needed)
exact_reps = True  # should the number of repetitions be exact or ok to exceed by 1?

block_structures = (
    np.array([1, 2, 3]),
    np.array([1, 3, 2]),
    np.array([2, 2, 2]),
    np.array([3, 2, 1]),
)

# Repeat 4 times (by stacking), then shuffle rows
repeated_blocks = np.vstack([block_structures for _ in range(4)])
np.random.shuffle(repeated_blocks)
SET_SIZE_TO_KEY_LIST = {
    2: np.vstack(
        (
            np.array([1, 0, 1]),
            np.array([1, 1, 0]),
            np.array([0, 1, 1]),
            np.array([1, 0, 1]),
        )
    ),
}

In [5]:
from pathlib import Path

from tqdm import trange

COLNAMES = [
    "stim",
    "correct_key",
    "set_size",
    "block",
    "img_folder",
    "condition",
]


class RLWMSequenceMaker:
    def __init__(
        self,
        exp_type,
        num_reps,
        block_structure,
        num_conditions=2,
        exact_reps=False,
        to_dir=None,
        num_keys=3,
        max_stims=6,
    ):
        self.exp_type = exp_type
        self.num_reps = num_reps
        self.num_conditions = num_conditions
        self.exact_reps = exact_reps
        self.num_keys = num_keys
        self.max_stims = max_stims
        self.block_structure = list(block_structure)
        self.num_blocks = len(self.block_structure)
        self.set_size_to_key_list = SET_SIZE_TO_KEY_LIST

    def make_sequences(self):
        """Generate one CSV / DataFrame per seq. Returns all DataFrame."""

        blocks = list(self.block_structure)
        block_rules = self._make_block_rules(blocks)
        stim_sets = np.random.permutation(self.num_blocks) + 2

        block_dfs = []
        for block_i, (ns, condition) in enumerate(blocks):
            prototype = self._make_seq_prototype(ns)
            correct_keys = np.vectorize(block_rules[block_i].get)(prototype - 1)

            block_dfs.append(
                pd.DataFrame(
                    {
                        "stim": prototype,
                        "correct_key": correct_keys,
                        "set_size": ns,
                        "block": block_i + 1,
                        "img_folder": stim_sets[block_i],
                        "condition": condition,
                    }
                )
            )

        return pd.concat(block_dfs, ignore_index=True)[COLNAMES]

    def _make_block_rules(self, blocks):
        """stim_idx (0-based) -> correct key, per block."""
        key_pools = {
            ns: shuffle_along_axis(keys, axis=1).tolist()
            for ns, keys in self.set_size_to_key_list.items()
        }

        block_rules = []
        for ns, _ in blocks:
            counts = key_pools[ns].pop()
            keys_for_stims = [
                key_i for key_i in range(self.num_keys) for _ in range(counts[key_i])
            ]
            block_rules.append(dict(enumerate(keys_for_stims)))
        return block_rules

    def _make_block_stimuli(self, blocks):
        """stim_idx (0-based) -> image number (1-based), per block."""
        return [
            dict(enumerate((np.random.permutation(self.max_stims) + 1)[:ns]))
            for ns, _ in blocks
        ]

    def _make_seq_prototype(self, set_size):
        """Stimulus indices 1..set_size, reshuffled each repetition cycle."""
        cycles = [
            shuffled(np.arange(1, set_size + 1)) for _ in range(self.num_reps + 1)
        ]
        return np.hstack(cycles)

In [51]:
# starts with 2 two because the 1st folder is practice.
np.random.permutation(2) + 2

array([3, 2])

# Checks

In [9]:
for i in np.arange(1, 2):
    df = pd.read_csv(f"{DIR_NAME}/seq{i}_learning.csv")

    print(df.groupby(["block"]).size())

block
1    96
2    96
dtype: int64
